In [ ]:
import pandas as pd
pd.options.plotting.backend ='plotly'
import plotly.express as px

In [ ]:
df1 = pd.read_csv('../clean_data/data_1_political_final.csv')
df2 = pd.read_csv('../clean_data/data_2_political_final.csv')
df = pd.concat([df1, df2])

# Remove duplicatas
df = df.drop_duplicates(subset=list(df.columns).remove('id'))

# Dataset length
print(len(df))

# Number of users
print(len(df['user'].unique()))

In [ ]:
# Interactions by type
df['type'].hist()

In [ ]:
# Languages
df.loc[~df['language'].isin(['fr', 'en']), 'language']  = 'Other'
df['language'].hist()

In [ ]:
# Interactions by user
count_pays = df['user'].value_counts().reset_index()
count_pays.columns = ['user', 'nb_observations']

count_pays = count_pays.sort_values('nb_observations').reset_index(drop=True)

fig = px.line(
    count_pays,
    x=count_pays.index,
    y='nb_observations',
    markers=False,
    title="Nombre d'interactions par utilisateur (ordonné)"
)

fig.update_xaxes(showticklabels=True, title="Utilisateurs")
fig.update_yaxes(title="Nombre d'interactions")

fig.show()

In [ ]:
# Conversion en datetime
df['date'] = pd.to_datetime(df['date'])

obs_par_semaine = (
    df
    .set_index('date')
    .resample('W')
    .agg(
        nb_observations=('id', 'size'),
        nb_user_uniques=('user', 'nunique')
    )
    .reset_index()
)

# Plot
fig = px.line(
    obs_par_semaine,
    x='date',
    y=['nb_observations', 'nb_user_uniques'],
    markers=False,
    title="Nombre d'interactions et d'utilisateurs uniques par semaine"
)

fig.show()

In [ ]:
# Assurer le bon format de date
df['date'] = pd.to_datetime(df['date'])

# Calcul de la durée de vie par utilisateur
duree_vie = (
    df
    .groupby('user')['date']
    .agg(['min', 'max'])
)

# Durée de vie en jours
duree_vie['duree_jours'] = (duree_vie['max'] - duree_vie['min']).dt.days

# Statistiques
moyenne = duree_vie['duree_jours'].mean()
q25 = duree_vie['duree_jours'].quantile(0.25)
q75 = duree_vie['duree_jours'].quantile(0.75)

print(f"Durée de vie moyenne : {moyenne:.2f} jours")
print(f"Quantile 25% : {q25:.2f} jours")
print(f"Quantile 75% : {q75:.2f} jours")